# Modeling - Extra Trees

First nonlinear tree baseline using grouped well validation. This notebook keeps the same simple test-safe feature set as Ridge so the comparison mostly reflects model family behavior, not feature drift.

## 1. Setup

Imports, paths, constants, and well bundles. (Run context.)

In [ ]:
"Imports, paths, constants, and train wells."
import sys
import time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    for parent in ROOT.parents:
        if (parent / "src").is_dir():
            ROOT = parent
            break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

from src.evaluation.metrics import metrics_summary
from src.evaluation.validation import group_well_folds, known_zone_mask, target_zone_mask
from src.io.loaders import discover_wells, load_all_wells
from src.modeling.trees import make_tree_model
from src.preprocessing.tabular import make_simple_feature_frame, sample_training_rows
from src.utils.paths import get_artifacts_dir, get_data_dir
from src.utils.run_bundle import RunBundle
from src.utils.seeds import set_global_seeds

set_global_seeds(42)
MODEL_NAME = "extra_trees"
MODEL_LABEL = "Extra Trees"
N_SPLITS = 3
MAX_TRAIN_ROWS_PER_FOLD = 30000
PARAMS = {
    "n_estimators": 80,
    "max_depth": 20,
    "min_samples_leaf": 6,
    "max_features": 0.9,
    "n_jobs": -1,
    "random_state": 42
}

train_dir = get_data_dir("train")
figures_dir = get_artifacts_dir("figures")
tables_dir = get_artifacts_dir("tables")
models_dir = get_artifacts_dir("models") / "trees"
for directory in [figures_dir, tables_dir, models_dir]:
    directory.mkdir(parents=True, exist_ok=True)

wells = discover_wells(train_dir)
well_bundle = load_all_wells(train_dir, wells=wells)
folds = group_well_folds(wells, n_splits=N_SPLITS)

print(f"model: {MODEL_LABEL}")
print(f"train wells: {len(wells)}")
print(f"folds: {len(folds)}")


## 2. Feature And Artifact Helpers

Build train/validation arrays from known-zone and target-zone rows. (Feature matrix and artifact utilities.)

In [ ]:
"Helpers for grouped folds, figures, and comparison rows."
def train_valid_arrays(well_bundle, train_wells, valid_wells):
    train_frames = []
    valid_frames = []
    valid_targets = []
    valid_meta = []

    for well in train_wells:
        df = well_bundle[well]["horizontal"]
        mask = known_zone_mask(df)
        feat = make_simple_feature_frame(df).loc[mask]
        feat["target"] = df.loc[mask, "TVT"].to_numpy(dtype=float)
        train_frames.append(feat)

    for well in valid_wells:
        df = well_bundle[well]["horizontal"]
        mask = target_zone_mask(df)
        feat = make_simple_feature_frame(df).loc[mask]
        valid_frames.append(feat)
        valid_targets.append(df.loc[mask, "TVT"].to_numpy(dtype=float))
        valid_meta.extend([well] * int(mask.sum()))

    train_df = pd.concat(train_frames, axis=0)
    valid_df = pd.concat(valid_frames, axis=0)
    y_train = train_df.pop("target").to_numpy(dtype=float)
    y_valid = np.concatenate(valid_targets).astype(float)

    medians = train_df.median(numeric_only=True)
    X_train = train_df.fillna(medians).to_numpy(dtype=float)
    X_valid = valid_df.fillna(medians).to_numpy(dtype=float)
    return X_train, y_train, X_valid, y_valid, valid_meta, list(train_df.columns), medians


def append_model_comparison(row):
    path = tables_dir / "model_comparison.csv"
    if path.exists():
        table = pd.read_csv(path)
        table = pd.concat([table, pd.DataFrame([row])], ignore_index=True)
    else:
        table = pd.DataFrame([row])
    table = table.drop_duplicates(["model_family", "model_name"], keep="last")
    table.to_csv(path, index=False)
    return path


def save_residual_strip(y_true, y_pred, path, title):
    width, height = 920, 360
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.text((24, 18), title, fill="black")
    residual = np.asarray(y_pred) - np.asarray(y_true)
    if residual.size > 1400:
        idx = np.linspace(0, residual.size - 1, 1400).astype(int)
        residual = residual[idx]
    lo, hi = np.percentile(residual, [1, 99])
    bound = max(abs(float(lo)), abs(float(hi)), 1.0)
    mid = height // 2
    draw.line((40, mid, width - 30, mid), fill="#444444")
    for i, value in enumerate(residual):
        x = 40 + int(i * (width - 80) / max(len(residual) - 1, 1))
        y = mid - int((float(value) / bound) * 120)
        color = "#d95f02" if value > 0 else "#1b9e77"
        draw.line((x, mid, x, y), fill=color)
    draw.text((24, height - 34), f"sampled residuals; clipped at +/-{bound:.2f}", fill="black")
    image.save(path)
    return path


def save_importance_bars(feature_columns, importances, path, title, top_n=14):
    pairs = sorted(zip(feature_columns, importances), key=lambda item: item[1], reverse=True)[:top_n]
    width, height = 920, 420
    image = Image.new("RGB", (width, height), "white")
    draw = ImageDraw.Draw(image)
    draw.text((24, 18), title, fill="black")
    max_value = max([float(v) for _, v in pairs] + [1e-9])
    left, top, bar_h = 210, 64, 22
    for i, (label, value) in enumerate(pairs):
        y = top + i * 25
        draw.text((24, y + 3), label[:25], fill="black")
        bar_w = int((width - left - 80) * float(value) / max_value)
        draw.rectangle((left, y, left + bar_w, y + bar_h), fill="#4c78a8")
        draw.text((left + bar_w + 8, y + 3), f"{float(value):.4f}", fill="black")
    image.save(path)
    return path


## 3. Grouped CV Fit

Fit the tree model on known-zone rows from training wells and score held-out target-zone rows. (Nonlinear validation.)

In [ ]:
"Grouped tree validation loop."
start = time.time()
fold_metric_rows = []
prediction_rows = []
feature_columns = None
last_medians = None
last_model = None

for fold_idx, (train_wells, valid_wells) in enumerate(folds, start=1):
    X_train, y_train, X_valid, y_valid, valid_meta, feature_columns, medians = train_valid_arrays(
        well_bundle, train_wells, valid_wells
    )
    X_train, y_train = sample_training_rows(X_train, y_train, MAX_TRAIN_ROWS_PER_FOLD)
    model = make_tree_model(MODEL_NAME, PARAMS).fit(X_train, y_train)
    pred = model.predict(X_valid)
    summary = metrics_summary(y_valid, pred)
    summary.update({
        "fold": fold_idx,
        "model": MODEL_NAME,
        "n_train_rows": len(y_train),
        "n_valid_rows": len(y_valid),
        "n_valid_wells": len(valid_wells),
    })
    fold_metric_rows.append(summary)
    prediction_rows.append(pd.DataFrame({
        "fold": fold_idx,
        "wellname": valid_meta,
        "y_true": y_valid,
        "y_pred": pred,
        "residual": pred - y_valid,
    }))
    last_model = model
    last_medians = medians
    print(f"fold {fold_idx}: rmse={summary['rmse']:.4f}, rows={len(y_valid)}")

fold_metrics = pd.DataFrame(fold_metric_rows)
predictions = pd.concat(prediction_rows, ignore_index=True)
summary = {
    "rmse_mean": float(fold_metrics["rmse"].mean()),
    "rmse_std": float(fold_metrics["rmse"].std(ddof=0)),
    "mae_mean": float(fold_metrics["mae"].mean()),
    "r2_mean": float(fold_metrics["r2"].mean()),
}
train_time_s = time.time() - start

fold_metrics


## 4. Final Fit And Artifacts

Refit on a deterministic known-zone sample, export metrics, residuals, feature importances, and a run bundle. (Reportable outputs.)

In [ ]:
"Final fit, artifact export, and run bundle."
all_train_wells = wells
X_all, y_all, _, _, _, feature_columns, medians = train_valid_arrays(well_bundle, all_train_wells[:-1], all_train_wells[-1:])
X_all, y_all = sample_training_rows(X_all, y_all, MAX_TRAIN_ROWS_PER_FOLD)
final_model = make_tree_model(MODEL_NAME, PARAMS).fit(X_all, y_all)

config = {
    "model_family": "trees",
    "model": MODEL_NAME,
    "params": PARAMS,
    "validation": "3-fold grouped by well; train known-zone rows; score target-zone rows",
    "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD,
}
with RunBundle(f"tree_{MODEL_NAME}", config) as run:
    metrics_path = tables_dir / f"model__{MODEL_NAME}_metrics__{run.run_id}.csv"
    residual_path = figures_dir / f"model__{MODEL_NAME}_residuals__{run.run_id}.png"
    importance_path = figures_dir / f"model__{MODEL_NAME}_feature_importance__{run.run_id}.png"
    model_path = models_dir / f"model__{MODEL_NAME}__{run.run_id}.pkl"

    fold_metrics.to_csv(metrics_path, index=False)
    save_residual_strip(predictions["y_true"].to_numpy(), predictions["y_pred"].to_numpy(), residual_path, f"{MODEL_LABEL} residuals")
    save_importance_bars(feature_columns, final_model.feature_importances_, importance_path, f"{MODEL_LABEL} feature importance")

    import pickle
    with open(model_path, "wb") as handle:
        pickle.dump(final_model, handle)

    run.add_input(str(train_dir))
    run.add_output(str(metrics_path))
    run.add_output(str(residual_path))
    run.add_output(str(importance_path))
    run.add_output(str(model_path))
    run.add_metric("cv_rmse_mean", summary["rmse_mean"])
    run.add_metric("cv_rmse_std", summary["rmse_std"])
    run.add_metric("cv_mae_mean", summary["mae_mean"])
    run.add_metric("cv_r2_mean", summary["r2_mean"])
    run.set_notes(f"{MODEL_LABEL} baseline on simple test-safe tabular features.")

comparison_path = append_model_comparison({
    "run_id": run.run_id,
    "model_family": "trees",
    "model_name": MODEL_NAME,
    "cv_rmse_mean": summary["rmse_mean"],
    "cv_rmse_std": summary["rmse_std"],
    "train_time_s": float(train_time_s),
    "n_features": len(feature_columns),
    "notes": f"{MODEL_LABEL}; grouped well CV; deterministic row cap",
})

print("run_id:", run.run_id)
print("summary:", summary)
print("metrics:", metrics_path)
print("figures:", residual_path, importance_path)
print("comparison:", comparison_path)
